<a href="https://www.kaggle.com/code/abhishekgodara/cafa-6-protein-prediction-score-0-377?scriptVersionId=292269648" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
import os
import gc
import pandas as pd
import numpy as np
from collections import defaultdict
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. OPTIMIZED OBO PARSER
# ==========================================
def parse_obo_parents(go_obo_path):
    """Parse OBO file with minimal memory footprint"""
    print(f"[1/5] Parsing OBO Ontology...")
    term_parents = {}
    roots = {'GO:0003674', 'GO:0008150', 'GO:0005575'}
    
    with open(go_obo_path, "r") as f:
        current_id = None
        for line in f:
            line = line.strip()
            if line == "[Term]":
                current_id = None
            elif line.startswith("id: GO:"):
                current_id = line[4:].strip()
            elif line.startswith("is_a: GO:"):
                if current_id:
                    pid = line.split()[1]
                    if current_id not in term_parents:
                        term_parents[current_id] = set()
                    term_parents[current_id].add(pid)
            elif line.startswith("relationship: part_of GO:"):
                if current_id:
                    parts = line.split()
                    if len(parts) >= 3:
                        pid = parts[2]
                        if current_id not in term_parents:
                            term_parents[current_id] = set()
                        term_parents[current_id].add(pid)
    
    return term_parents, roots

def get_ancestors_map(term_parents):
    """Build ancestor map with iterative DFS to avoid recursion depth issues"""
    print("[1/5] Building Ancestor Map...")
    ancestors = {}
    visited = set()
    
    for term in tqdm(term_parents.keys(), desc="Building ancestors"):
        if term in visited:
            continue
            
        stack = [term]
        term_ancestors = set()
        
        while stack:
            current = stack[-1]
            
            if current in ancestors:
                term_ancestors.update(ancestors[current])
                stack.pop()
                continue
                
            if current not in visited:
                visited.add(current)
                
                # Get direct parents
                parents = term_parents.get(current, set())
                unprocessed_parents = [p for p in parents if p not in ancestors]
                
                if unprocessed_parents:
                    stack.extend(unprocessed_parents)
                else:
                    # All parents processed, compute ancestors
                    all_ancestors = set()
                    for p in parents:
                        all_ancestors.add(p)
                        all_ancestors.update(ancestors.get(p, set()))
                    
                    ancestors[current] = all_ancestors
                    term_ancestors.update(all_ancestors)
                    stack.pop()
            else:
                stack.pop()
    
    return ancestors

# ==========================================
# 2. OPTIMIZED PROCESSING LOGIC
# ==========================================
def process_predictions(df, ancestors_map, roots):
    """Process predictions with optimized memory usage and scoring"""
    print("[3/5] Processing Predictions...")
    
    # Pre-filter: Remove predictions for terms not in ontology
    valid_terms = set(ancestors_map.keys()).union(
        *[ancestors_map[k] for k in ancestors_map]
    ).union(roots)
    
    # Filter dataframe in place
    mask = df['go_term'].isin(valid_terms)
    df = df[mask].copy()
    
    # Convert scores to float32 to save memory
    df['score'] = df['score'].astype(np.float32)
    
    # Group by protein - using pandas groupby is often faster than manual dict
    print("[3/5] Grouping predictions by protein...")
    grouped = df.groupby('protein_id')
    
    new_rows = []
    
    # Process each protein group
    for pid, group in tqdm(grouped, total=len(grouped.groups), desc="Processing proteins"):
        # Convert to dictionary for faster lookups
        term_scores = dict(zip(group['go_term'], group['score']))
        
        # Initialize with current scores
        final_scores = term_scores.copy()
        
        # --- A. UPWARD PROPAGATION (Parents get max of children) ---
        for term, score in term_scores.items():
            if score > 0:
                # Get ancestors and propagate
                for anc in ancestors_map.get(term, set()):
                    current = final_scores.get(anc, 0.0)
                    if score > current:
                        final_scores[anc] = score
        
        # --- B. ENSURE ROOTS ARE PRESENT ---
        if final_scores:
            for root in roots:
                final_scores[root] = 1.0
        
        # --- C. IMPROVED NORMALIZATION STRATEGY ---
        # Separate root and non-root terms
        non_root_scores = {}
        for term, score in final_scores.items():
            if term not in roots:
                non_root_scores[term] = score
        
        if non_root_scores:
            # Apply softmax-like normalization instead of simple scaling
            scores_array = np.array(list(non_root_scores.values()), dtype=np.float32)
            
            # Apply temperature scaling to sharpen distribution
            temperature = 0.7  # Lower temperature = sharper distribution
            scaled_scores = np.exp(scores_array / temperature)
            normalized = scaled_scores / scaled_scores.sum()
            
            # Boost top predictions while preserving relative ordering
            max_score = scores_array.max()
            if max_score < 0.8:
                boost_factor = 0.8 / max_score
                boosted_scores = np.minimum(1.0, scores_array * boost_factor)
            else:
                boosted_scores = scores_array
            
            # Blend original and boosted scores (70% boosted, 30% normalized)
            final_non_root = 0.7 * boosted_scores + 0.3 * normalized
            
            # Update non-root scores
            for (term, _), new_score in zip(non_root_scores.items(), final_non_root):
                if new_score >= 0.001:  # Filter threshold
                    new_rows.append((pid, term, float(new_score)))
        
        # Add roots
        for root in roots:
            new_rows.append((pid, root, 1.0))
    
    return pd.DataFrame(new_rows, columns=['protein_id', 'go_term', 'score'])

# ==========================================
# 3. CHUNKED PROCESSING FOR LARGE FILES
# ==========================================
def process_large_file_chunked(input_path, ancestors_map, roots, chunksize=1000000):
    """Process very large files in chunks to avoid memory issues"""
    print(f"[2/5] Loading submission in chunks...")
    
    all_chunks = []
    
    # First pass: count rows for progress bar
    total_rows = 0
    for chunk in pd.read_csv(input_path, sep='\t', header=None, 
                           names=['protein_id', 'go_term', 'score', 'key'],
                           chunksize=chunksize, usecols=[0, 1, 2]):
        total_rows += len(chunk)
    
    # Reset and process
    processed_rows = 0
    chunk_iterator = pd.read_csv(input_path, sep='\t', header=None,
                               names=['protein_id', 'go_term', 'score', 'key'],
                               chunksize=chunksize, usecols=[0, 1, 2])
    
    for chunk in tqdm(chunk_iterator, total=total_rows/chunksize, desc="Processing chunks"):
        processed = process_predictions(chunk, ancestors_map, roots)
        all_chunks.append(processed)
        processed_rows += len(chunk)
        
        # Optional: Clear memory periodically
        if len(all_chunks) >= 5:
            combined = pd.concat(all_chunks, ignore_index=True)
            all_chunks = [combined]
            gc.collect()
    
    # Combine all chunks
    final_df = pd.concat(all_chunks, ignore_index=True)
    return final_df

# ==========================================
# 4. MAIN PIPELINE WITH OPTIMIZATIONS
# ==========================================
def main():
    # Paths
    OBO_PATH = "/kaggle/input/cafa-6-protein-function-prediction/Train/go-basic.obo"
    SUBMISSION_INPUT = '/kaggle/input/cafa6-protein-function-enhanced-nb-v2/submission.tsv'
    SUBMISSION_OUTPUT = 'submission.tsv'
    
    # 1. Load Ontology
    term_parents, roots = parse_obo_parents(OBO_PATH)
    ancestors_map = get_ancestors_map(term_parents)
    
    # 2. Load and Process Submission
    print(f"[2/5] Loading submission...")
    
    # Estimate file size to decide chunking strategy
    file_size = os.path.getsize(SUBMISSION_INPUT)
    USE_CHUNKING = file_size > 100 * 1024 * 1024  # 100MB threshold
    
    if USE_CHUNKING:
        print(f"Large file detected ({file_size/1024/1024:.1f} MB), using chunked processing...")
        final_df = process_large_file_chunked(SUBMISSION_INPUT, ancestors_map, roots)
    else:
        submission = pd.read_csv(SUBMISSION_INPUT, sep='\t', header=None, 
                               names=['protein_id', 'go_term', 'score', 'key'],
                               usecols=[0, 1, 2])
        final_df = process_predictions(submission, ancestors_map, roots)
    
    # 3. Sort and Save
    print(f"[4/5] Sorting and saving {len(final_df):,} rows...")
    
    # Sort by protein_id and score (descending)
    final_df = final_df.sort_values(['protein_id', 'score'], 
                                   ascending=[True, False],
                                   kind='mergesort')  # mergesort is stable
    
    # Group by protein and keep top N predictions per protein to reduce file size
    # This helps if you have too many predictions per protein
    MAX_PREDS_PER_PROTEIN = 1500
    if 'protein_id' in final_df.columns:
        final_df = final_df.groupby('protein_id').head(MAX_PREDS_PER_PROTEIN)
    
    # Save to file
    final_df.to_csv(SUBMISSION_OUTPUT, sep='\t', index=False, header=False)
    
    # 4. Statistics
    print(f"[✅] Done. Saved to {SUBMISSION_OUTPUT}")
    print(f"Statistics:")
    print(f"  - Total predictions: {len(final_df):,}")
    print(f"  - Unique proteins: {final_df['protein_id'].nunique():,}")
    print(f"  - Unique GO terms: {final_df['go_term'].nunique():,}")
    print(f"  - Score range: [{final_df['score'].min():.3f}, {final_df['score'].max():.3f}]")
    print(f"  - Mean score: {final_df['score'].mean():.3f}")
    
    # Sample output
    print("\nTop predictions sample:")
    print(final_df.head(10).to_string(index=False))

if __name__ == "__main__":
    main()